# rapids_build_backend

rapids-build-backend is a PEP 517 build backend that wraps another backend and reads its own settings from the
`[tool.rapids-build-backend]` table of `pyproject.toml`. The examples below wrap setuptools' backend, so setuptools has to
be installed next to it:

```bash
pip install rapids-build-backend setuptools
```

Every example works in its own temporary project directory.

## Example 1: Read the `[tool.rapids-build-backend]` table with `Config`

In [ ]:
# Example 1: Read the [tool.rapids-build-backend] table with Config
import os
import tempfile
import textwrap

from rapids_build_backend.config import Config

PYPROJECT = textwrap.dedent("""\
    [build-system]
    requires = ["rapids-build-backend", "setuptools"]
    build-backend = "rapids_build_backend.build"

    [project]
    name = "demo-pkg"
    version = "0.1.0"

    [tool.rapids-build-backend]
    build-backend = "setuptools.build_meta"
    dependencies-file = "dependencies.yaml"
    matrix-entry = "cuda_suffixed=false"
    commit-files = []
    requires = []
""")

with tempfile.TemporaryDirectory() as tmp:
    with open(os.path.join(tmp, "pyproject.toml"), "w") as f:
        f.write(PYPROJECT)
    config = Config(dirname=tmp)
    print("build_backend:    ", config.build_backend)
    print("dependencies_file:", config.dependencies_file)
    print("matrix_entry:     ", config.matrix_entry)
    print("disable_cuda:     ", config.disable_cuda)
    print("commit_files:     ", config.commit_files)

    assert config.build_backend == "setuptools.build_meta"
    assert config.dependencies_file == "dependencies.yaml"
    assert config.matrix_entry == "cuda_suffixed=false"
    assert config.disable_cuda is False          # not set in the table, so the default applies
    assert list(config.commit_files) == []
    assert list(config.requires) == []

    # A pyproject.toml without the table is rejected
    with open(os.path.join(tmp, "pyproject.toml"), "w") as f:
        f.write('[project]\nname = "plain"\nversion = "0.1.0"\n')
    try:
        Config(dirname=tmp)
    except RuntimeError as e:
        print("Without the table:", e)
    else:
        raise AssertionError("Config accepted a pyproject.toml without [tool.rapids-build-backend]")

## Example 2: Override options with `RAPIDS_*` environment variables and config settings

In [ ]:
# Example 2: Override options with RAPIDS_* environment variables and config settings
import os
import tempfile

from rapids_build_backend.config import Config

with tempfile.TemporaryDirectory() as tmp:
    with open(os.path.join(tmp, "pyproject.toml"), "w") as f:
        f.write(
            '[project]\nname = "demo-pkg"\nversion = "0.1.0"\n\n'
            '[tool.rapids-build-backend]\nbuild-backend = "setuptools.build_meta"\ndisable-cuda = false\n'
        )

    # Config settings (pip's --config-settings) override the table
    from_settings = Config(dirname=tmp, config_settings={"rapidsai.disable-cuda": "true"})
    assert from_settings.disable_cuda is True

    # Environment variables override both; save and restore the caller's values
    saved = {k: os.environ.get(k) for k in ("RAPIDS_DISABLE_CUDA", "RAPIDS_MATRIX_ENTRY")}
    try:
        os.environ["RAPIDS_DISABLE_CUDA"] = "true"
        os.environ["RAPIDS_MATRIX_ENTRY"] = "cuda=12.5;arch=x86_64"
        config = Config(dirname=tmp, config_settings={"rapidsai.disable-cuda": "false"})
        print("disable_cuda from RAPIDS_DISABLE_CUDA:", config.disable_cuda)
        print("matrix_entry from RAPIDS_MATRIX_ENTRY:", config.matrix_entry)
        assert config.disable_cuda is True
        assert config.matrix_entry == "cuda=12.5;arch=x86_64"
        assert config.build_backend == "setuptools.build_meta"   # not overridable

        os.environ["RAPIDS_DISABLE_CUDA"] = "yes"   # boolean options accept only "true" or "false"
        try:
            Config(dirname=tmp).disable_cuda
        except ValueError as e:
            print("Rejected:", e)
        else:
            raise AssertionError("RAPIDS_DISABLE_CUDA=yes was accepted")
    finally:
        for k, v in saved.items():
            if v is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = v

## Example 3: Import the PEP 517 hooks from `rapids_build_backend.build`

In [ ]:
# Example 3: Import the PEP 517 hooks from rapids_build_backend.build
# The build module reads pyproject.toml from the working directory when it is imported, so import it
# from inside a temporary project and restore the working directory afterwards.
import importlib
import os
import sys
import tempfile

PYPROJECT = (
    '[build-system]\nrequires = ["rapids-build-backend", "setuptools"]\nbuild-backend = "rapids_build_backend.build"\n\n'
    '[project]\nname = "demo-pkg"\nversion = "0.1.0"\n\n'
    '[tool.rapids-build-backend]\nbuild-backend = "setuptools.build_meta"\n'
    'dependencies-file = "dependencies.yaml"\nmatrix-entry = "cuda_suffixed=false"\n'
    'disable-cuda = true\ncommit-files = []\nrequires = []\n'
)

hooks = ["build_wheel", "build_sdist", "get_requires_for_build_wheel",
         "get_requires_for_build_sdist", "prepare_metadata_for_build_wheel"]

cwd = os.getcwd()
with tempfile.TemporaryDirectory() as tmp:
    with open(os.path.join(tmp, "pyproject.toml"), "w") as f:
        f.write(PYPROJECT)
    os.chdir(tmp)
    try:
        sys.modules.pop("rapids_build_backend.build", None)
        build = importlib.import_module("rapids_build_backend.build")
        for hook in hooks:
            print(f"{hook}: {callable(getattr(build, hook, None))}")
            assert callable(getattr(build, hook, None)), hook
        assert build.config.build_backend == "setuptools.build_meta"
        assert build.backend.__name__ == "setuptools.build_meta"
    finally:
        os.chdir(cwd)
print(f"All {len(hooks)} hooks are callable")

## Example 4: Compute wheel and sdist build requirements from `dependencies.yaml`

In [ ]:
# Example 4: Compute wheel and sdist build requirements from dependencies.yaml
# get_requires_for_build_* runs rapids-dependency-file-generator on the project's dependencies.yaml,
# adds the result to the build requirements, and puts pyproject.toml back the way it was.
import importlib
import os
import sys
import tempfile

PYPROJECT = (
    '[project]\nname = "demo-pkg"\nversion = "0.1.0"\n\n'
    '[tool.rapids-build-backend]\nbuild-backend = "setuptools.build_meta"\n'
    'dependencies-file = "dependencies.yaml"\nmatrix-entry = "cuda_suffixed=false"\n'
    'disable-cuda = true\ncommit-files = []\nrequires = []\n'
)
DEPENDENCIES = """\
files:
  py_build:
    output: pyproject
    pyproject_dir: .
    extras:
      table: tool.rapids-build-backend
      key: requires
    includes: [build_tools]
channels: [conda-forge]
dependencies:
  build_tools:
    common:
      - output_types: [pyproject, requirements]
        packages: [cython>=3.0, scikit-build-core]
"""

cwd = os.getcwd()
with tempfile.TemporaryDirectory() as tmp:
    with open(os.path.join(tmp, "pyproject.toml"), "w") as f:
        f.write(PYPROJECT)
    with open(os.path.join(tmp, "dependencies.yaml"), "w") as f:
        f.write(DEPENDENCIES)
    os.chdir(tmp)
    try:
        sys.modules.pop("rapids_build_backend.build", None)
        build = importlib.import_module("rapids_build_backend.build")
        wheel_requires = build.get_requires_for_build_wheel({})
        sdist_requires = build.get_requires_for_build_sdist({})
        with open("pyproject.toml") as f:
            restored = f.read()
        leftovers = sorted(os.listdir("."))
    finally:
        os.chdir(cwd)

print("wheel build requires:", wheel_requires)
print("sdist build requires:", sdist_requires)
assert sdist_requires == ["cython>=3.0", "scikit-build-core"], sdist_requires
# With setuptools as the wrapped backend, "wheel" is added for bdist_wheel
assert wheel_requires == ["cython>=3.0", "scikit-build-core", "wheel"], wheel_requires
assert restored == PYPROJECT
assert leftovers == ["dependencies.yaml", "pyproject.toml"], leftovers